# 🔬 LLM Benchmark Platform — GPU Worker (Colab T4)

**Before running:** Runtime → Change runtime type → GPU (T4)

This notebook benchmarks GPTQ and AWQ models on a free Colab T4 GPU
and saves results as JSON files for the platform.

In [ ]:
# Step 1: Clone the repo
!git clone https://github.com/YOUR_USERNAME/llm-benchmark-platform.git
%cd llm-benchmark-platform

In [ ]:
# Step 2: Install GPU dependencies
!pip install transformers==4.40.0 accelerate==0.29.2 optimum==1.19.0
!pip install auto-gptq==0.7.1 --extra-index-url https://huggingface.github.io/autogptq-index/whl/cu121/
!pip install autoawq==0.2.4
!pip install pydantic rich psutil duckdb pandas

In [ ]:
# Step 3: Verify GPU
!nvidia-smi

In [ ]:
# Step 4: Run GPTQ benchmark — Mistral 7B
import sys
sys.path.insert(0, '.')

from src.workers.gpu_worker import run_benchmark
from src.core.models import QuantizationType

result = run_benchmark(
    model_id     = 'TheBloke/Mistral-7B-v0.1-GPTQ',
    quantization = QuantizationType.GPTQ_4BIT,
    n_runs       = 3,
    output_dir   = 'data/raw',
)
print(f'Result: {result.tokens_per_sec:.2f} tok/s')

In [ ]:
# Step 5: Run AWQ benchmark — Mistral 7B
result = run_benchmark(
    model_id     = 'TheBloke/Mistral-7B-v0.1-AWQ',
    quantization = QuantizationType.AWQ_4BIT,
    n_runs       = 3,
    output_dir   = 'data/raw',
)

In [ ]:
# Step 6: Run Phi-3-mini AWQ
result = run_benchmark(
    model_id     = 'microsoft/Phi-3-mini-4k-instruct-AWQ',
    quantization = QuantizationType.AWQ_4BIT,
    n_runs       = 3,
    output_dir   = 'data/raw',
)

In [ ]:
# Step 7: Download all raw JSON results
import os
from google.colab import files

raw_files = list(os.listdir('data/raw'))
print(f'Found {len(raw_files)} result files:')
for f in raw_files:
    print(f'  {f}')
    files.download(f'data/raw/{f}')

print('\nDownload these files and place them in data/raw/ on your local machine.')
print('Then run: python scripts/load_seed_data.py --raw-dir data/raw/')